# Designing a IIR Filter

In [ ]:
import numpy as np
import ipywidgets as widgets
import matplotlib.pyplot as plt
from scipy.signal import sosfilt, sosfreqz, sos2zpk
from rp.lia.config import calc_iir_coeffs
from rp.lia.processing import calc_r_phi, reconstruct_signal

# Plotting settings
# plt.rcParams['axes.grid'] = True
plt.rcParams['grid.linestyle'] = '--'  # Grid line style
plt.rcParams['grid.linewidth'] = 0.5  # Grid line width
plt.rcParams['axes.titlesize'] = 14  # Title font size
plt.rcParams['axes.labelsize'] = 12  # Axis label font size
plt.rcParams['xtick.labelsize'] = 10  # X tick label font size
plt.rcParams['ytick.labelsize'] = 10  # Y tick label font size
plt.rcParams['legend.fontsize'] = 10  # Legend font size
plt.rcParams['xtick.direction'] = 'in'  # X tick direction
plt.rcParams['ytick.direction'] = 'in'  # Y tick direction
plt.rcParams['xtick.top'] = True  # Show ticks on top
plt.rcParams['ytick.right'] = True

## Signal Generation

In [ ]:
SIGNAL = None
@widgets.interact(
    fs_MHz = 125*np.asarray([1,1/2,1/4,1/5,1/8,1/10,1/25]),
    f_kHz=np.round(np.linspace(1,2000,2000,endpoint=False)),
    ref_scaling = (0, 16, 1),
    p_sig=(0, 360, 10),
    periods = (1, 1000, 1),
    noise=(0, 1, 0.1),
    plot_period = (1, 100, 1),
    write_file = [False, True], 
    write_samples = (10, 100_000, 1),
)
def plot_LIA(
    fs_MHz=31.25,
    f_kHz=500,
    ref_scaling=14,
    p_sig=0,
    periods=2,
    noise=0.2,
    plot_period=2,
    write_file=False,
    write_samples=10000,
):
    fs = fs_MHz * 1e6
    f = f_kHz * 1e3

    # Calculate samples needed for given frequency, periods and sampling rate
    samples_per_period = int(fs / f)
    samples = samples_per_period * periods

    # Generate the time vector
    t = np.arange(samples) / fs

    # Generate the sine signal
    sig = np.sin(2 * np.pi * f * t + np.deg2rad(p_sig))
    sig_noise = sig + np.random.normal(0, noise, samples)
    # Scale signal to simulate hardware ADC
    sin_ref = (2**ref_scaling - 1) * np.sin(2 * np.pi * f * t)
    cos_ref = (2**ref_scaling - 1) * np.cos(2 * np.pi * f * t)
    
    time_us = np.asarray(t)*1e6
    
    # Create global dict for signal
    global SIGNAL
    SIGNAL = {
        "t" : t,
        "sig": sig,
        "sig_noise": sig_noise,
        "samples_per_period": samples_per_period,
        "fs_MHz": fs_MHz,
        "f_kHz" : f_kHz,
        "ref_scaling": ref_scaling,
        "sin_ref": sin_ref,
        "cos_ref": cos_ref,
        "t_us":time_us
    }
    
    # Save signal to tv file
    if write_file:
        with open("./test_data/i_data.tv", "w") as f:
            mix_sig = sig_noise * sin_ref
            for i in range(write_samples):
                s = int(mix_sig[i])
                # convert to two's complement
                if s < 0:
                    s = (1 << 32) + s
                # convert to hex
                f.write(f"{s:08X}\n")
                
        with open("./test_data/q_data.tv", "w") as f:
            mix_sig = sig_noise * cos_ref
            for i in range(write_samples):
                s = int(mix_sig[i])
                # convert to two's complement
                if s < 0:
                    s = (1 << 32) + s
                # convert to hex
                f.write(f"{s:08X}\n")
                

    plt.figure(figsize=(12, 3))
    plt.plot(time_us,sig_noise, label="Signal + Noise")
    plt.plot(time_us,sig, label="Signal")
    #plt.xlim(0, plot_period * samples_per_period /(time_us[1]-time_us[0]))
    plt.legend()
    plt.xlabel("time in µs")
    print(f"Number of samples: {samples}")
    print(f"Number of samples per period: {samples_per_period}")
    print(f"Sampling frequency: {fs}")

## Filtering

## Filter Simulation with preconfigured filter coefficients

In [ ]:
fs_MHz = SIGNAL["fs_MHz"]
f_kHz = SIGNAL["f_kHz"]

@widgets.interact(
    fs_MHz=125*np.asarray([1,1/2,1/4,1/5,1/8,1/10,1/25]),
    f_kHz=np.round(np.linspace(1,2000,2000,endpoint=False)),
    fc_kHz=(1, 1_000, 1),
    filt_type=['butter', 'cheby1', 'cheby2'],
    attn_dB=(10, 120, 10),
    sig_periods=(10, 1_000, 10),
    order=(1, 100, 1),
    coeff_bits=(1, 32, 1),
    frac_bits=(1, 32, 1),
)
def plot_LIA(
    fs_MHz=fs_MHz,
    f_kHz=f_kHz,
    fc_kHz=250,
    filt_type='butter',
    attn_dB=60,
    sig_periods=200,
    order=2,
    coeff_bits=25,
    frac_bits=23,
):

    fs = fs_MHz * 1e6
    f  = f_kHz * 1e3
    fc = fc_kHz * 1e3

    # Get parameters from global dict
    t = SIGNAL["t"]
    signal = SIGNAL["sig_noise"]
    samples_per_period = SIGNAL["samples_per_period"]
    ref_scaling = SIGNAL["ref_scaling"]
    sin_ref = SIGNAL["sin_ref"]
    cos_ref = SIGNAL["cos_ref"]
    t_us = SIGNAL["t_us"]
    
    # Mixing the signal with the reference signal
    I = signal * sin_ref
    Q = signal * cos_ref

    # Filtering with floating point coefficients
    coeff = calc_iir_coeffs(order, fc, fs, filt_type=filt_type, attn_dB=attn_dB)
    I_filt = sosfilt(coeff, I)
    Q_filt = sosfilt(coeff, Q)
    
    # Filtering with quantized coefficients
    coeff_quan = np.round(coeff * 2**frac_bits) / 2**frac_bits
    I_filt_quantized = sosfilt(coeff_quan, I)
    Q_filt_quantized = sosfilt(coeff_quan, Q)

    # Calculate amplitude and phase and reconstruct the signal
    r, phi = calc_r_phi(I_filt, Q_filt)
    print(f"reconstruct with SR: {fs}Hz for freq {f}Hz")
    out_signal = reconstruct_signal(r, phi, f, fs, ref_scaling)
    
    # Calculate the frequency response of the filter
    w, h = sosfreqz(coeff, worN=8000)
    freq = w / (2 * np.pi) * fs
    gain = np.abs(h)
    # Find the passband edge (0.99 gain) and stopband edge (0.01 gain)
    pb_edge = freq[np.where(gain <= 0.99)[0][0]]
    sb_edge = freq[np.where(gain <= 0.01)[0][0]]
    f_cutoff = freq[np.where(gain <= 0.707)[0][0]]

    ################################################################################################
    # Plot filter response and properties
    ################################################################################################
    # Plotting the Frequency Response with Passband and Stopband Edges
    # plt.figure(figsize=(20, 5))
    # plt.subplot(131)
    # plt.plot(freq, gain, 'b', label=f'Filter Response (fc = {fc} Hz = {fc/(fs/2)} * fs)')
    # plt.axvline(pb_edge, color='k', linestyle='--', label=f'Passband Edge at {int(pb_edge)} Hz')
    # plt.axvline(sb_edge, color='k', linestyle='--', label=f'Stopband Edge at {int(sb_edge)} Hz')
    # plt.axvline(f_cutoff, color='red', linestyle='--',label=f'-3 dB Cutoff at {int(f_cutoff)} Hz')
    # plt.axvspan(pb_edge, sb_edge, color='yellow', alpha=0.3, label=f'Transition Band Width: {int(sb_edge - pb_edge)} Hz')
    # plt.title("Butterworth Filter Frequency Response (Gain)")
    # plt.xlabel('Frequency (Hz)')
    # plt.ylabel('Gain')
    # plt.xlim(0, 2*sb_edge)
    # plt.legend()

    # # Plot Impuls Response
    # plt.subplot(132)
    # impulse = np.zeros(5000)
    # impulse[0] = 1
    # impulse_response = sosfilt(coeff, impulse)
    # plt.plot(impulse_response)
    # plt.title("Impulse Response")
    # plt.xlabel("Sample (n)")

    # # Plot zeros and poles using tf2zpk
    # plt.subplot(133)
    # plt.title("Zeros and Poles")
    # plt.xlabel('Real')
    # plt.ylabel('Imaginary')
    # theta = np.linspace(0, 2*np.pi, 100) # unit circle
    # x = np.cos(theta)
    # y = np.sin(theta)
    # plt.plot(x, y, color='black', linestyle='-')

    # # Zeros and poles of original coefficients
    # z, p, k = sos2zpk(coeff)
    # plt.scatter(np.real(z), np.imag(z), marker='o', color='blue', label='Zeros')
    # plt.scatter(np.real(p), np.imag(p), marker='x', color='red', label='Poles')
    # # Zeros and poles of quantized coefficients
    # z_q, p_q, k_q = sos2zpk(coeff_quan)
    # plt.scatter(np.real(z_q), np.imag(z_q), marker='o', color='green', label='Zeros Quantized')
    # plt.scatter(np.real(p_q), np.imag(p_q), marker='x', color='orange', label='Poles Quantized')
    # plt.xlim(-1.5, 1.5)
    # plt.ylim(-1.5, 1.5)
    # plt.legend()
    
    # Assign IIR filter parameters to global variable
    global IIR
    IIR = {"order": order, "coeff": coeff, "coeff_bits": coeff_bits, "frac_bits": frac_bits}

    ################################################################################################
    # Plot filtered signal and reconstructed signal
    ################################################################################################
    plt.figure(figsize=(20, 5))
    samples = samples_per_period * sig_periods

    plt.subplot(131)
    plt.title('In-phase Component')
    plt.plot(t_us,I[:samples], color="tab:green", label="unfiltered")
    plt.plot(t_us,I_filt[:samples], color="tab:red", label="filtered")
    plt.plot(t_us,I_filt_quantized[:samples],
             color="tab:blue", label="filtered quantized")
    plt.legend()

    plt.subplot(132)
    plt.title('Quadrature Component')
    plt.plot(t_us,Q[:samples], color="tab:green", label="unfiltered")
    plt.plot(t_us,Q_filt[:samples], color="tab:red", label="filtered")
    plt.plot(t_us,Q_filt_quantized[:samples],
             color="tab:blue", label="filtered quantized")
    plt.legend()

    plt.subplot(133)
    plt.title('Signal and Reconstructed Signal')
    plt.plot(t_us,signal[:samples], color="tab:green", label="Input signal")
    plt.plot(t_us,out_signal[:samples], color="tab:red", label="Reconstructed_signal")
    plt.legend()

    # When plotting using cheby2 use following parameters as an example (no example for cheby1)
    # f = 125_000
    # fs = 125_000_000
    # fc = 100_000
    # filt_type = 'cheby2'
    # attn_dB = 60
    # sig_periods = 100
    # order = 6
    # coeff_bits = 25
    # frac_bits = 23
    

## Writing Filter Coefficients to file

These coeficients are used inside the Vivado simulation of the IIR filter
simulation. The coefficients are loaded into the simulation and are sent to the
filter's internal registers.

In [ ]:
# Run the cell above first to get the IIR filter parameters
coeff = IIR["coeff"]
ORDER = IIR["order"]
SOS_NUM = len(coeff)
SOS_COEFF_NUM = len(coeff[0])
COEFF_WIDTH = IIR["coeff_bits"]
COEFF_FRAC_WIDTH = IIR["frac_bits"]
# Number of IIR module's internal registers 
IIR_REGS = 32

print(f"filter order: {ORDER}, sos sections: {SOS_NUM}")
print(f"coeff_width: {COEFF_WIDTH}, frac_bits: {COEFF_FRAC_WIDTH}\n")

# Create test_data folder if it does not exist
import os
if not os.path.exists("test_data"):
    os.makedirs("test_data")
    
# Write the coefficients to a file
with open("./test_data/iir_coeff_data.tv", "w") as f:
    # Give information about the number of sos
    f.write(f"{SOS_NUM:08X}\n")
    regs = 1    
    
    # Write the coefficients to a file
    for i in range(SOS_NUM):
        print(f"second order section {i}:")
        for j in range(SOS_COEFF_NUM):
            # skip a0 coefficient
            if j != 3:
                # convert to two's complement
                quantization = 2**COEFF_FRAC_WIDTH
                s = round(coeff[i][j] * quantization)
                s_error = coeff[i][j] - s/quantization
                if s < 0:
                    s = (1 << COEFF_WIDTH) + s

                # convert to hex and write to file
                print(f"0x{s:08X} <- {coeff[i][j]:>20.12e} | error: {s_error:>20.12e}")
                f.write(f"{s:08X}\n")
                regs += 1
                
    if regs < IIR_REGS:
        for i in range(IIR_REGS - regs):
            f.write("00000000\n")